# Phase 3: Data Preprocessing & Augmentations

This notebook defines data transformation matrices, resizes inputs to standard 224x224 sizes, applies training augmentations (rotations, horizontal flips) to expand leaf patterns, and constructs validation PyTorch `DataLoader` pipelines.

In [1]:
# Path and import setup
from pathlib import Path
import sys
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != 'ai' and PROJECT_ROOT.parent != PROJECT_ROOT:
    if (PROJECT_ROOT / 'ai').exists():
        PROJECT_ROOT = PROJECT_ROOT / 'ai'
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import utils

### 1. Define Transform Composites

Uses standard ImageNet means and standard deviations to align inputs for transfer learning.

In [2]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Pre-processing transformations
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(utils.config.IMAGE_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(utils.config.IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

print("✅ Augmentation transforms initialized successfully.")

✅ Augmentation transforms initialized successfully.


### 2. Construct PyTorch DataLoaders

In [3]:
train_dir = utils.config.TRAIN_DIR
val_dir = utils.config.VAL_DIR

if train_dir.exists() and val_dir.exists():
    train_dataset = datasets.ImageFolder(root=str(train_dir), transform=train_transforms)
    val_dataset = datasets.ImageFolder(root=str(val_dir), transform=val_transforms)
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=utils.config.BATCH_SIZE,
        shuffle=True,
        num_workers=utils.config.NUM_WORKERS,
        pin_memory=torch.cuda.is_available()
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=utils.config.BATCH_SIZE,
        shuffle=False,
        num_workers=utils.config.NUM_WORKERS,
        pin_memory=torch.cuda.is_available()
    )
    
    print(f"✅ Training Loader: {len(train_loader)} batches ({len(train_dataset)} images)")
    print(f"✅ Validation Loader: {len(val_loader)} batches ({len(val_dataset)} images)")
else:
    print("⚠️ Dataset splits not found. Please complete environment bring-up and verification first.")

✅ Training Loader: 1358 batches (43444 images)
✅ Validation Loader: 340 batches (10861 images)
